# GameTheory-18 : Capstone — ce qui survit au changement de representation

**Navigation** : [<< 17-MultiAgent-RL](GameTheory-17-MultiAgent-RL-Python.ipynb) | [Index](README.md) | [Capstone 19 >>](GameTheory-19-Capstone-Dynamics-Python.ipynb) · [Version C#](GameTheory-18-Capstone-Metajeux-CSharp.ipynb) · sous-série [Metajeux](Metajeux/README.md)

**Kernel** : Python 3 (cpu)

## Concept

Ce carnet est le **capstone** de la sous-serie `Metajeux/` : il n'apporte aucun theoreme
nouveau, il **relie** les six carnets qui la composent autour d'une seule question.

> Quand on change l'espace de representation d'un jeu, qu'est-ce qui survit ?

La sous-serie a mesure, et la reponse est en trois temps :

- **Composer** puis **casser** : composer des reponses locales ne survit pas a l'ambiguite
  d'une reponse locale multivoque. La composition predit un **ensemble**, pas un point ;
- **Abstraire** : l'abstraction ne survit pas du tout a l'equilibre, mais la **dette**
  qu'elle contracte est mesurable, et elle se lit des deux cotes du jeu ;
- **Temoigner** puis **transposer** : un **verdict** survit au changement de substrat, une
  **mesure** ne survit pas.

Ce qui survit, en un mot : **la methode**, pas le nombre. Chaque chiffre de ce carnet est
**rejoue** ici, pas recopie ; quand un chiffre change de valeur parce que le substrat
change, le carnet le dit.

## Prerequis

- Les six carnets de la sous-serie : [01](Metajeux/Metajeux-01-Open-Games-Lentilles-Python.ipynb) · [02](Metajeux/Metajeux-02-Casser-la-Composition-Python.ipynb) · [03](Metajeux/Metajeux-03-Abstraction-a-Dette-Python.ipynb) ·
  [04](Metajeux/Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) · [05](Metajeux/Metajeux-05-Temoins-Impossibilite-Python.ipynb) · [06](Metajeux/Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb)
- Notions de composition de fonctions, de strategie mixte et d'exploitabilite, de BFS sur un graphe.

## Duree estimee : 40 minutes

## Limites

Les experiences 1 a 3 rejouent des **protocoles condenses** des carnets sources : memes
instances, memes formules, memes seeds, mais des extraits plus courts. Quand un carnet source
execute un recensement exhaustif sur 518 400 paires, ce carnet en rejoue le **resultat** et
en **refait** une partie ; le detail reste dans le carnet source, cite a chaque fois.


## Objectifs d'apprentissage

1. **Relier** les six carnets de la sous-serie autour d'une seule question, et nommer ce qui
   se transporte d'une representation a l'autre — et ce qui ne se transporte pas.
2. **Rejouer** trois protocoles mesures (composer/casser, abstraire, temoigner) et verifier
   que les chiffres obtenus ici sont ceux des carnets sources.
3. **Verifier** qu'une conclusion de representation est un **verdict** (transportable) ou une
   **mesure** (locale au substrat), en faisant echouer une generalisation volontairement.
4. **Separer** ce qui est calcule par le code de ce qui est certifie par une preuve : la
   premiere partie est rejouable, la seconde est lue.

In [1]:
# -*- coding: utf-8 -*-
"""Capstone Metajeux : trois experiences rejouees (composer/casser, abstraire, temoigner)."""

import itertools
import random
import time
from collections import deque

import numpy as np

RNG = np.random.default_rng(20261115)

print(f"numpy={np.__version__}")
print("Capstone Metajeux : trois experiences, six carnets de la sous-serie.")
print("Chaque chiffre est REJOUE ici, jamais recopie.")

numpy=2.5.3
Capstone Metajeux : trois experiences, six carnets de la sous-serie.
Chaque chiffre est REJOUE ici, jamais recopie.


## Ce que la sous-serie demontre

Les six carnets de `Metajeux/` ne sont pas six sujets : ils sont **six moments d'un meme
geste**, celui de la representation. Le capstone les rejoue dans cet ordre.

| Carnet | Ce qu'il etablit | Ce que ce carnet en reprend |
|---|---|---|
| [Metajeux-01](Metajeux/Metajeux-01-Open-Games-Lentilles-Python.ipynb) | les lois d'une lentille, la composition des meilleures reponses (Hedges), trois dynamiques de boucle | le regime **univoque** de l'experience 1 |
| [Metajeux-02](Metajeux/Metajeux-02-Casser-la-Composition-Python.ipynb) | la **casse** : composer des selections echoue des que la reponse locale est multivoque | l'experience 1 (la table des deviations) |
| [Metajeux-03](Metajeux/Metajeux-03-Abstraction-a-Dette-Python.ipynb) | la **dette d'abstraction**, chiffree, et une preuve Lean qui la recoupe | l'experience 2 (la courbe de dette) |
| [Metajeux-04](Metajeux/Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) | le constructeur et le verificateur **separes** (Loi II) | l'experience 3a (la certification) |
| [Metajeux-05](Metajeux/Metajeux-05-Temoins-Impossibilite-Python.ipynb) | le **temoin d'impossibilite** certifie sur 576 chambres | l'experience 3a (les cas A, B, C) |
| [Metajeux-06](Metajeux/Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb) | le **second substrat** 3x2, ou le verdict 2x2 ne survit pas | l'experience 3b (le recensement) |

La colonne de droite est aussi la **couverture** du capstone : les six carnets de la
sous-serie sont consommes, chacun par au moins une experience.

## Trois experiences de dix minutes

1. **Composer, puis casser** (Metajeux/Metajeux-01-Open-Games-Lentilles-Python.ipynb + Metajeux/Metajeux-02-Casser-la-Composition-Python.ipynb) — la composition tient-elle ?
2. **Abstraire, puis mesurer la dette** (Metajeux/Metajeux-03-Abstraction-a-Dette-Python.ipynb) — que perd-on ?
3. **Temoigner, puis changer de substrat** (Metajeux/Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb + Metajeux/Metajeux-05-Temoins-Impossibilite-Python.ipynb + Metajeux/Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb) — qu'est-ce qui
   passe d'une representation a l'autre ?

## Experience 1 — Composer, puis casser

**Ce que fait l'experience.** Deux jeux ouverts en serie : J1 choisit `x`, J2 repond `y`,
et le couplage est bilatere (`u2` depend de `x`, `u1` depend de `y`). On compose les
meilleures reponses locales, puis on demande au **jeu compose** — par enumeration des
deviations — si le profil obtenu est un equilibre.

**Le protocole tient en trois ingredients**, et l'experience en isole un par un :

- l'**univoquite** : la reponse locale de J1 a un unique maximum ;
- la **multivocite** : le maximum local est un plateau (ici, deux actions ex aequo par
  symetrie exacte au milieu de l'intervalle) ;
- le **couplage** : le choix de J1 revient a J1 par la reponse de J2.

Le carnet source [Metajeux-02](Metajeux/Metajeux-02-Casser-la-Composition-Python.ipynb)
montre que les deux premiers sont **inoffensifs separement** et que c'est leur conjonction
qui brise la selection. On rejoue les trois situations ici.

In [2]:
# === Experience 1 : composer des reponses locales, puis demander au jeu compose ===
# Protocole condense de Metajeux-02 (cellules 3 a 8) : memes formules, memes instances.
ACTIONS_X = (0, 1, 2)
ACTIONS_Y = (0, 1, 2)
C_PLATEAU, R_COUPLAGE = 0.5, 0.3

def u1(x, y):
    """Plateau exact en x=0 et x=1 (c=0.5), couple en retour par r*y."""
    return -(x - C_PLATEAU) ** 2 + R_COUPLAGE * y

def u2(x, y):
    """Reponse de J2 : maximum unique en y = x (couplage dans l'autre sens)."""
    return -(y - x) ** 2

def br2(x):
    """Meilleure reponse de J2 voyant x."""
    return max(ACTIONS_Y, key=lambda y: u2(x, y))

def br1_ensemble():
    """Correspondance locale de J1 (y exogene) : l'ENSEMBLE des optima, et les valeurs."""
    vals = [u1(x, 0.0) for x in ACTIONS_X]
    vmax = max(vals)
    return [x for x in ACTIONS_X if abs(u1(x, 0.0) - vmax) < 1e-12], vals

def est_nash_compose(x, y, u1_fn):
    """Nash du jeu compose : aucune deviation profitable, J1 anticipant y' = br2(x')."""
    ok1 = all(u1_fn(x, y) >= u1_fn(xp, br2(xp)) - 1e-12 for xp in ACTIONS_X)
    ok2 = all(u2(x, y) >= u2(x, yp) - 1e-12 for yp in ACTIONS_Y)
    return ok1 and ok2

# --- Contre-fait 1 : couplage SANS plateau (reponse locale univoque) ---
def u1_univoque(x, y):
    return -(x - 1.0) ** 2

x_u, y_u = 1, br2(1)
print("[Contre-fait 1 : couplage SANS plateau, reponse locale univoque]")
print(f"  profil compose : ({x_u}, {y_u}) | Nash du compose : {est_nash_compose(x_u, y_u, u1_univoque)}")

# --- Le regime casse : plateau multivoque + couplage ---
br1_A, vals_locales = br1_ensemble()
profils_composes = [(x, br2(x)) for x in br1_A]
selection_naive = min(br1_A)

print()
print("[Regime casse : plateau multivoque + couplage]")
print(f"  valeurs locales de u1 (y exogene) : {[f'{v:+.3f}' for v in vals_locales]}")
print(f"  BR locale de J1 = {br1_A}  (MULTIVOQUE : plateau par symetrie exacte c=0.5)")
print(f"  composition naive (plus petit argmax) : ({selection_naive}, {br2(selection_naive)})")
print(f"  composition des correspondances : {profils_composes}")
print()
print(f"{'Profil prevu':<14} {'u1(x, y)':>9} {'meilleure deviation J1':>23} {'ecart':>8}  Verdict")
print("-" * 78)
for (x, y) in profils_composes:
    ecart = max(u1(xp, br2(xp)) for xp in ACTIONS_X) - u1(x, y)
    x_best = max(ACTIONS_X, key=lambda xp: u1(xp, br2(xp)))
    verdict = "NON-NASH du compose (J1 devie)" if ecart > 1e-9 else "equilibre du compose"
    print(f"({x}, {y}){'':<7} {u1(x, y):9.3f} ({x_best}, {br2(x_best)}){'':<12} {ecart:+8.3f}  {verdict}")

# --- Contre-fait 2 : plateau SANS couplage (r = 0) ---
u1_r0 = lambda x, y: -(x - C_PLATEAU) ** 2
ecarts_r0 = [max(u1_r0(xp, br2(xp)) for xp in ACTIONS_X) - u1_r0(x, br2(x)) for (x, y) in profils_composes]
print()
print(f"[Contre-fait 2 : plateau SANS couplage (r=0)] ecarts = "
      f"[{ecarts_r0[0]:+.3f}, {ecarts_r0[1]:+.3f}] -> les deux profils survivent (equilibres faibles)")

[Contre-fait 1 : couplage SANS plateau, reponse locale univoque]
  profil compose : (1, 1) | Nash du compose : True

[Regime casse : plateau multivoque + couplage]
  valeurs locales de u1 (y exogene) : ['-0.250', '-0.250', '-2.250']
  BR locale de J1 = [0, 1]  (MULTIVOQUE : plateau par symetrie exacte c=0.5)
  composition naive (plus petit argmax) : (0, 0)
  composition des correspondances : [(0, 0), (1, 1)]

Profil prevu    u1(x, y)  meilleure deviation J1    ecart  Verdict
------------------------------------------------------------------------------
(0, 0)           -0.250 (1, 1)               +0.300  NON-NASH du compose (J1 devie)
(1, 1)            0.050 (1, 1)               +0.000  equilibre du compose

[Contre-fait 2 : plateau SANS couplage (r=0)] ecarts = [+0.000, +0.000] -> les deux profils survivent (equilibres faibles)


### Lecture du resultat — la composition predit un ensemble, pas un point

Trois mesures, une seule conclusion.

- **Contre-fait 1** (couplage sans plateau) : le profil compose `(1, 1)` est bien un
  equilibre du jeu compose, verifie par enumeration des deviations. Le couplage, seul, ne
  casse rien.
- **Regime casse** : la reponse locale de J1 est le plateau `{0, 1}` — les valeurs locales
  sont `-0.250`, `-0.250`, `-2.250`, l'ex aequo est produit par la symetrie exacte
  `c = 0.5`. La composition **naive** (prendre le plus petit element du plateau) propose
  `(0, 0)`, et le jeu compose le **refuse** : J1, anticipant `y = x`, gagne `+0.300` en
  deviant vers `x = 1`. Le profil `(1, 1)` de la composition des correspondances, lui, est
  un equilibre.
- **Contre-fait 2** (plateau sans couplage, `r = 0`) : les deux ecarts tombent a `+0.000` —
  les deux profils survivent, indifferentiable.

**Ce qui survit au changement de representation, ici :** la *correspondance* — l'ensemble
des meilleures reponses, avec son contexte — et non la *selection*, le point qu'on en
extrait. Un carnet qui lit « la meilleure reponse de la composition est la composition des
meilleures reponses » comme une egalite de points a tort : c'est une egalite d'ensembles, et
la lecture fonctionnelle ne survit que si la reponse locale est univoque.

## Experience 2 — Abstraire, puis mesurer la dette

**Ce que fait l'experience.** Un jeu `G` trop grand pour etre lu d'un coup : six duels 2x2
independants. On le **reduit** a trois etats abstraits (un duel moyen par bloc), on
**resout** l'abstrait exactement, on **retransporte** la strategie, puis on mesure ce que
la strategie retransportee vaut **dans le jeu d'origine** — jamais dans l'abstrait.

C'est le retournement du carnet [Metajeux-03](Metajeux/Metajeux-03-Abstraction-a-Dette-Python.ipynb) :
la question n'est plus « cette representation compresse-t-elle ? » mais « qu'ai-je le droit
d'oublier, et combien cela me coute ? ». On rejoue ici la construction, la resolution
exacte, la retransportation, et la **courbe de dette** sur la chaine de raffinement.

In [3]:
# === Experience 2 : abstraire, resoudre, retransporter, mesurer la dette DANS le jeu d'origine ===
# Protocole condense de Metajeux-03 (cellules 1, 3, 7, 12) : meme graine, memes matrices.
N_STATES, N_ACTIONS = 6, 2

random.seed(42)
PAYOFFS = {}
for state in range(N_STATES):
    for a_pair in itertools.product(range(N_ACTIONS), repeat=N_ACTIONS):
        PAYOFFS[(state, a_pair)] = random.randint(-3, 3)

def duel_matrix(s):
    """Matrice du duel de l'etat s : M_s[a1][a2] = gain du joueur 1."""
    return [[PAYOFFS[(s, (a1, a2))] for a2 in range(N_ACTIONS)] for a1 in range(N_ACTIONS)]

# Controle de reproduction : les matrices affichees par le carnet source.
MATRICES_SOURCE = [
    [[2, -3], [-3, 2]], [[-1, -2], [-2, -2]], [[2, -3], [2, 2]],
    [[1, -3], [1, 0]], [[-3, -3], [-3, -2]], [[-2, 1], [1, -3]],
]
reproduit = all(duel_matrix(s) == MATRICES_SOURCE[s] for s in range(N_STATES))
print(f"Jeu G : somme de {N_STATES} duels 2x2 independants (zero-sum), graine 42.")
print(f"  matrices regenerees identiques a celles du carnet source : {reproduit}")
assert reproduit, "les matrices ne sont pas celles du carnet source"
for s in range(N_STATES):
    print(f"  M_{s} = {duel_matrix(s)}")

def solve_2x2(m):
    """Equilibre EXACT d'un duel 2x2 zero-sum, par ENUMERATION DE SUPPORTS.
    Rend (p, q, val) : p[a1] = proba de ligne, q[a2] = proba de colonne, val = valeur."""
    a, b, c, d = m[0][0], m[0][1], m[1][0], m[1][1]
    for (i, j) in itertools.product(range(N_ACTIONS), repeat=N_ACTIONS):
        if (m[i][j] == max(m[x][j] for x in range(N_ACTIONS))
                and m[i][j] == min(m[i][y] for y in range(N_ACTIONS))):
            p = [1.0, 0.0] if i == 0 else [0.0, 1.0]
            q = [1.0, 0.0] if j == 0 else [0.0, 1.0]
            return p, q, m[i][j]
    den = a - b - c + d
    p_star, q_star = (d - c) / den, (d - b) / den
    assert 0 <= p_star <= 1 and 0 <= q_star <= 1
    return [p_star, 1 - p_star], [q_star, 1 - q_star], (a * d - b * c) / den

def abstract_matrix(block):
    """Duel abstrait d'un bloc : MOYENNE des duels des etats fusionnes."""
    return [[sum(PAYOFFS[(s, (a1, a2))] for s in block) / len(block)
             for a2 in range(N_ACTIONS)] for a1 in range(N_ACTIONS)]

def solve_and_transport(partition):
    """Solve exact de G_tilde pour la partition donnee, puis retransport vers G."""
    sigma_tilde = {k: solve_2x2(abstract_matrix(block)) for k, block in enumerate(partition)}
    where = {s: k for k, block in enumerate(partition) for s in block}
    sig1 = lambda s, a: sigma_tilde[where[s]][0][a]
    sig2 = lambda s, a: sigma_tilde[where[s]][1][a]
    value_tilde = sum(sigma_tilde[k][2] for k in sigma_tilde)
    return sig1, sig2, value_tilde

def exploitabilite(sig1, sig2):
    """expl = max_pi1 u_1(pi_1, sigma_2) + max_pi2 u_2(sigma_1, pi_2) ; nulle ssi Nash."""
    br1 = sum(max(sum(sig2(s, a2) * PAYOFFS[(s, (a1, a2))] for a2 in range(N_ACTIONS))
                  for a1 in range(N_ACTIONS)) for s in range(N_STATES))
    br2 = sum(max(sum(sig1(s, a1) * (-PAYOFFS[(s, (a1, a2))]) for a1 in range(N_ACTIONS))
                  for a2 in range(N_ACTIONS)) for s in range(N_STATES))
    return br1 + br2, br1, br2

VALUE_G = sum(solve_2x2(duel_matrix(s))[2] for s in range(N_STATES))
PARTITION_3 = [(0, 1), (2, 3), (4, 5)]

sig1, sig2, v_tilde = solve_and_transport(PARTITION_3)
expl, br1_max, br2_max = exploitabilite(sig1, sig2)

print()
print(f"Valeur exacte de G (somme des {N_STATES} duels) : v(G) = {VALUE_G:.4f}")
print(f"Abstraction P3 (3 blocs de 2 etats) : facteur de reduction = {N_STATES / len(PARTITION_3):.1f}")
print(f"  v(G_tilde) = {v_tilde:.4f}")
print("Mesure DANS G de la strategie retransportee :")
print(f"  BR_1 = max_pi1 u_1(pi_1, sigma_2) = {br1_max:.4f}   (v(G) = {VALUE_G:.4f}, ecart = {br1_max - VALUE_G:+.4f})")
print(f"  BR_2 = max_pi2 u_2(sigma_1, pi_2) = {br2_max:.4f}   (-v(G) = {-VALUE_G:.4f}, ecart = {br2_max + VALUE_G:+.4f})")
print(f"  exploitabilite = {expl:.4f} -> la strategie retransportee n'est PAS un equilibre de G")

# --- La courbe de dette : taille de l'abstraction contre dette ---
PARTITIONS = [
    (6, [(0,), (1,), (2,), (3,), (4,), (5,)]),
    (4, [(0, 1), (2,), (3,), (4, 5)]),
    (3, [(0, 1), (2, 3), (4, 5)]),
    (2, [(0, 1, 2, 3), (4, 5)]),
]
print()
print("Courbe de dette (solve exact + retransport + exploitabilite dans G) :")
print(f"  {'|G_tilde|':>9}  {'v(G_tilde)':>10}  {'exploitabilite':>14}")
points = []
for n_abs, partition in PARTITIONS:
    s1, s2, v_t = solve_and_transport(partition)
    e, _, _ = exploitabilite(s1, s2)
    points.append((n_abs, e))
    print(f"  {n_abs:>9}  {v_t:>10.4f}  {e:>14.4f}")

# Sanity check : sans abstraction, l'exploitabilite est EXACTEMENT nulle.
assert abs(points[0][1]) < 1e-12, "P6 doit rendre 0 : solve exact + retransport identiques"
non_dec = all(points[i][1] <= points[i + 1][1] + 1e-9 for i in range(len(points) - 1))
plateau = abs(points[1][1] - points[2][1]) < 1e-9
forme = ("monotone non-decroissante" if non_dec else "NON monotone") + (
        " avec un palier P4 = P3" if plateau else "")
print(f"  forme mesuree : {forme}")


Jeu G : somme de 6 duels 2x2 independants (zero-sum), graine 42.
  matrices regenerees identiques a celles du carnet source : True
  M_0 = [[2, -3], [-3, 2]]
  M_1 = [[-1, -2], [-2, -2]]
  M_2 = [[2, -3], [2, 2]]
  M_3 = [[1, -3], [1, 0]]
  M_4 = [[-3, -3], [-3, -2]]
  M_5 = [[-2, 1], [1, -3]]

Valeur exacte de G (somme des 6 duels) : v(G) = -4.2143
Abstraction P3 (3 blocs de 2 etats) : facteur de reduction = 2.0
  v(G_tilde) = -1.8864
Mesure DANS G de la strategie retransportee :
  BR_1 = max_pi1 u_1(pi_1, sigma_2) = -2.8182   (v(G) = -4.2143, ecart = +1.3961)
  BR_2 = max_pi2 u_2(sigma_1, pi_2) = 4.7273   (-v(G) = 4.2143, ecart = +0.5130)
  exploitabilite = 1.9091 -> la strategie retransportee n'est PAS un equilibre de G

Courbe de dette (solve exact + retransport + exploitabilite dans G) :
  |G_tilde|  v(G_tilde)  exploitabilite
          6     -4.2143          0.0000
          4     -0.8864          1.9091
          3     -1.8864          1.9091
          2     -1.9342          6.4

### Lecture du resultat — la dette se lit des deux cotes

- **La reproduction est exacte** : les six duels regeneres par la graine 42 sont
  identiques, au signe pres, a ceux du carnet source ; la valeur exacte du jeu est
  `v(G) = -4.2143`, calculee par enumeration de supports, sans aucun processus iteratif.
- **L'abstraction ne survit pas** : la strategie resolue dans les trois etats abstraits,
  retransportee dans `G`, se fait exploiter des deux cotes (`BR_1 - v = +1.3961`,
  `BR_2 + v = +0.5130`). L'exploitabilite vaut `1.9091` : la dette n'est pas un phenomene
  symetrique, elle est **plus grande du cote du joueur qui perd le plus**.
- **La courbe de dette est lisible** : `0.0000` a six etats (sanity check — resoudre `G`
  puis retransporter ne change rien), `1.9091` a quatre etats, `1.9091` a trois etats, et
  plus encore a deux. La forme mesuree est **monotone non decroissante avec un palier
  P4 = P3** : a taille egale, la dette ne depend pas que de la taille mais de **quels**
  etats on fusionne (le carnet source montre qu'une fusion particuliere est gratuite).

**Ce qui survit au changement de representation, ici :** pas l'equilibre — **la dette**,
qui est une propriete transportable, chiffree, et lisible des deux cotes. C'est exactement
ce que le carnet [Metajeux-03](Metajeux/Metajeux-03-Abstraction-a-Dette-Python.ipynb) fait
recouper par une preuve Lean (voir la section « Ce que le lake Lean prouve » plus bas).

## Experience 3a — Temoigner : certifier l'impossibilite par la recherche

**Ce que fait l'experience.** Le dual du chemin minimal : prouver qu'**aucun** chemin plus
court n'existe. Sur les jeux ordinaux 2x2 de Robinson-Goforth, une chambre est une paire
d'ordres stricts sur 4 cases ; les generateurs echangeent deux cases adjacentes **d'un seul
cote** a la fois. Le graphe des chambres est donc le **produit cartesien** de deux copies
du permutoedre `S4`, et la distance s'y **decompose** :

```
d_chambre(G, H) = d_perm(row_G, row_H) + d_perm(col_G, col_H)
```

Le carnet [Metajeux-05](Metajeux/Metajeux-05-Temoins-Impossibilite-Python.ipynb) en tire un
**certificat** : si `d_row + d_col > k`, alors aucun chemin de longueur `<= k` n'existe —
l'impossibilite est un **verdict**, porte par une preuve, et verifiable par une recherche
independante. On rejoue ici la structure produit, le certificat sur trois cas, et le
contre-controle par BFS tronque.

In [4]:
# === Experience 3a : certifier un temoin d'impossibilite (structure produit + BFS tronque) ===
# Protocole condense de Metajeux-05 (cellules 2, 4, 6, 8, 10) : meme graphe, meme structure.
from itertools import permutations

def canonique(t):
    """Cle canonique d'un uplet : relabelage croissant des valeurs distinctes."""
    vals = sorted(set(t))
    return tuple(vals.index(v) + 1 for v in t)

def swap_valeurs_adjacentes(t, k):
    pk, pk1 = t.index(k), t.index(k + 1)
    l = list(t)
    l[pk], l[pk1] = l[pk1], l[pk]
    return tuple(l)

def swap_jeu(jeu, cote, k):
    row, col = jeu
    if cote == "ligne":
        return (swap_valeurs_adjacentes(row, k), col)
    return (row, swap_valeurs_adjacentes(col, k))

def adj_chambre(g):
    """6 voisins par chambre : 3 swaps de valeurs, sur chaque cote."""
    return [swap_jeu(g, cote, k) for cote in ("ligne", "colonne") for k in (1, 2, 3)]

def adj_perm(t):
    return {swap_valeurs_adjacentes(t, k) for k in (1, 2, 3)}

def bfs(depart, voisins_fn):
    """BFS exhaustive sur un graphe donne par fonction de voisinage."""
    dist = {depart: 0}
    q = deque([depart])
    while q:
        u = q.popleft()
        for v in voisins_fn(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

stricts = sorted({canonique(t) for t in permutations(range(1, 5))})
chambres = [(r, c) for r in stricts for c in stricts]
IDENTITE = ((1, 2, 3, 4), (1, 2, 3, 4))
RENVERS = ((4, 3, 2, 1), (4, 3, 2, 1))

print(f"Permutoedre : {len(stricts)} ordres stricts | Chambres : {len(chambres)} sommets")

# --- Structure produit : verification exhaustive ---
dist_perm = {o: bfs(o, adj_perm) for o in stricts}
dist_chambre = bfs(IDENTITE, adj_chambre)
ecarts = [(g, dist_chambre[g], dist_perm[IDENTITE[0]][g[0]] + dist_perm[IDENTITE[1]][g[1]])
          for g in chambres
          if dist_chambre[g] != dist_perm[IDENTITE[0]][g[0]] + dist_perm[IDENTITE[1]][g[1]]]
print(f"Verification exhaustive sur {len(chambres)} chambres : {len(ecarts)} ecarts")
print(f"Structure produit : {'EXACTE' if not ecarts else 'NON'}")
print(f"Diametre permutoedre : {max(max(d.values()) for d in dist_perm.values())} "
      f"| Diametre chambres : {max(dist_chambre.values())}")
assert not ecarts

# --- Le certificat analytique ---
D = {(o1, o2): dist_perm[o1][o2] for o1 in stricts for o2 in stricts}
non_sym = sum(1 for o1 in stricts for o2 in stricts if D[(o1, o2)] != D[(o2, o1)])
violations = sum(1 for o1, o2, o3 in itertools.product(stricts, repeat=3)
                 if D[(o1, o3)] > D[(o1, o2)] + D[(o2, o3)])
print(f"Table 24x24 : {non_sym} asymetries | inegalite triangulaire : {violations} violations "
      f"sur {24 ** 3} triplets")
assert violations == 0

def certifier(G, H, k_max):
    """Verdict d'impossibilite porte par le certificat (d_row, d_col, k_max)."""
    d_row = D[(G[0], H[0])]
    d_col = D[(G[1], H[1])]
    d_min = d_row + d_col
    impossible = d_min > k_max
    return {"verdict": "IMPOSSIBLE" if impossible else "POSSIBLE",
            "d_min": d_min,
            "preuve": f"d_perm(row) + d_perm(col) = {d_row} + {d_col} = {d_min} "
                      f"{'>' if impossible else '<='} {k_max}"}

H_B = ((1, 2, 3, 4), (3, 4, 1, 2))
H_C = ((1, 2, 4, 3), (1, 2, 3, 4))
for nom, G, H, k in [("CAS A (antipode)", IDENTITE, RENVERS, 5),
                     ("CAS B (ligne identique)", IDENTITE, H_B, 3),
                     ("CAS C (frontiere)", IDENTITE, H_C, 1)]:
    res = certifier(G, H, k)
    print(f"{nom}, k_max={k} : {res['verdict']} - {res['preuve']}")

# --- Contre-controle operationnel : BFS tronque ---
def bfs_tronque(depart, voisins_fn, profondeur_max):
    """Sommets atteignables en au plus profondeur_max pas (sans reconstruire le graphe)."""
    atteints = {depart}
    frontiere = {depart}
    for _ in range(profondeur_max):
        nouvelle = set()
        for u in frontiere:
            for v in voisins_fn(u):
                if v not in atteints:
                    atteints.add(v)
                    nouvelle.add(v)
        frontiere = nouvelle
        if not frontiere:
            break
    return atteints

atte_A = bfs_tronque(IDENTITE, adj_chambre, 5)
atte_B = bfs_tronque(IDENTITE, adj_chambre, 3)
atte_C = bfs_tronque(IDENTITE, adj_chambre, 1)
print(f"CAS A : {len(atte_A)} sommets a <= 5 pas, antipode present ? {RENVERS in atte_A}")
print(f"CAS B : {len(atte_B)} sommets a <= 3 pas, H_B present ? {H_B in atte_B}")
print(f"CAS C : {len(atte_C)} sommets a <= 1 pas, H_C present ? {H_C in atte_C}")
assert RENVERS not in atte_A and H_B not in atte_B and H_C in atte_C
print("Les deux voies sont d'accord : 2 IMPOSSIBLE, 1 POSSIBLE, verifies par la recherche.")

# --- Le seuil de bascule du verdict ---
seuil = next(k for k in range(0, 13) if certifier(IDENTITE, RENVERS, k)["verdict"] == "POSSIBLE")
print(f"Verdict POSSIBLE pour l'antipode a partir de k = {seuil} (distance reelle = 12) : bascule nette.")

Permutoedre : 24 ordres stricts | Chambres : 576 sommets
Verification exhaustive sur 576 chambres : 0 ecarts
Structure produit : EXACTE
Diametre permutoedre : 6 | Diametre chambres : 12
Table 24x24 : 0 asymetries | inegalite triangulaire : 0 violations sur 13824 triplets
CAS A (antipode), k_max=5 : IMPOSSIBLE - d_perm(row) + d_perm(col) = 6 + 6 = 12 > 5
CAS B (ligne identique), k_max=3 : IMPOSSIBLE - d_perm(row) + d_perm(col) = 0 + 4 = 4 > 3
CAS C (frontiere), k_max=1 : POSSIBLE - d_perm(row) + d_perm(col) = 1 + 0 = 1 <= 1
CAS A : 235 sommets a <= 5 pas, antipode present ? False
CAS B : 68 sommets a <= 3 pas, H_B present ? False
CAS C : 7 sommets a <= 1 pas, H_C present ? True
Les deux voies sont d'accord : 2 IMPOSSIBLE, 1 POSSIBLE, verifies par la recherche.
Verdict POSSIBLE pour l'antipode a partir de k = 12 (distance reelle = 12) : bascule nette.


### Lecture du resultat — un verdict, et sa preuve

- **La structure produit est exacte** : sur les 576 chambres, la distance du produit
  egale la somme des distances des deux cotes — **0 ecart**. Les diametres mesures sont `6`
  (permutoedre) et `12` (chambres), et la table 24x24 est symetrique et triangulaire
  (0 violation sur 13 824 triplets).
- **Le certificat donne le verdict, et sa preuve** : le cas A (l'antipode) est `IMPOSSIBLE`
  a `k = 5` parce que `6 + 6 = 12 > 5` ; le cas B est `IMPOSSIBLE` a `k = 3` parce que
  `0 + 4 = 4 > 3` ; le cas C est `POSSIBLE` a `k = 1` parce que `1 + 0 = 1 <= 1`.
- **La recherche confirme le certificat** : le BFS tronque atteint 235 sommets a 5 pas
  (l'antipode est absent), 68 a 3 pas (`H_B` absent), 7 a 1 pas (`H_C` present). Les deux
  voies — le raisonnement analytique et la recherche exhaustive — **concordent**.
- **La bascule est nette** : le verdict de l'antipode bascule de `IMPOSSIBLE` a `POSSIBLE`
  exactement a `k = 12`, la distance reelle. Aucun palier, aucune zone grise.

**Ce qui survit au changement de representation, ici :** le **verdict**. Un temoin
d'impossibilite reste vrai quand on change de representation, parce que son contenu n'est
pas une mesure mais une **structure** — la decomposition du produit, qui se verifie.

## Experience 3b — Changer de substrat : ce qui ne survit pas

**La question falsifiable.** Sur 2x2, le verdict mesure par le carnet
[Metajeux-04](Metajeux/Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) est que
traverser les murs (passer par des ex aequo) ne raccourcit **jamais** un chemin minimal.
Est-ce une propriete du jeu, ou une propriete du petit substrat ? Le carnet
[Metajeux-06](Metajeux/Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb) monte au
substrat **3x2 ordinal** — six cases par joueur, 720 tables strictes, 518 400 chambres
strictes — et **refute** la generalisation. On rejoue ici les deux recensements, dans le
meme modele ferme (fusion + scission des murs), pour comparer.

In [5]:
# === Experience 3b : changer de substrat (3x2 ordinal) -- le verdict survit-il ? ===
# Protocole condense de Metajeux-06 (cellules 3, 7, 9, 11, 13) : meme modele ferme.
from itertools import product, combinations

def ordres_faibles(n):
    """Tous les rangements faibles (avec ex aequo) de n cases, formes canoniques."""
    return sorted({canonique(t) for t in product(range(1, n + 1), repeat=n)})

def swap_adjacent(t, k):
    """Traversee de facette k<->k+1 : echange les cases portant les valeurs k et k+1."""
    pk, pk1 = t.index(k), t.index(k + 1)
    l = list(t)
    l[pk], l[pk1] = l[pk1], l[pk]
    return tuple(l)

def fusion(t, k):
    """Descente de mur : coalesce les valeurs k et k+1, forme canonique."""
    return canonique(tuple(k if v == k + 1 else v for v in t))

def scissions(t):
    """Remontee de mur : chaque bipartition non triviale d'un bloc d'ex aequo, dans les deux ordres."""
    res = []
    for v in sorted(set(t)):
        P = [i for i, x in enumerate(t) if x == v]
        m = len(P)
        if m < 2:
            continue
        for j in range(1, m):
            for S in combinations(P, j):
                s = set(S)
                res.append(canonique(tuple(
                    (v + 1 if x == v and i not in s else
                     (v if x == v else (x + 1 if x > v else x)))
                    for i, x in enumerate(t))))
    return res

def voisins_permutoedre(t, n):
    return [swap_adjacent(t, k) for k in range(1, n)]

def voisins_treillis(t, n):
    """Fusions (k et k+1 presents) + scissions -- le graphe de couverture du treillis."""
    res = []
    vals = set(t)
    for k in range(1, n):
        if k in vals and k + 1 in vals:
            res.append(fusion(t, k))
    res.extend(scissions(t))
    return res

def bfs_graphe(src, adj):
    """BFS sur un graphe donne par dictionnaire sommet -> voisinage."""
    dist = {src: 0}
    q = deque([src])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

OF4 = ordres_faibles(4)
ST4 = [t for t in OF4 if len(set(t)) == 4]
OF6 = ordres_faibles(6)
ST6 = [t for t in OF6 if len(set(t)) == 6]
print(f"Substrat 2x2 : ordres faibles = {len(OF4)} | stricts = {len(ST4)} | chambres = {len(ST4) ** 2}")
print(f"Substrat 3x2 : ordres faibles = {len(OF6)} | stricts = {len(ST6)} | chambres = {len(ST6) ** 2}")

graphs = {}
for n, OF, ST in ((4, OF4, ST4), (6, OF6, ST6)):
    adjP = {t: tuple(voisins_permutoedre(t, n)) for t in ST}
    adjC = {t: tuple(voisins_treillis(t, n)) for t in OF}
    asym = sum(1 for u, vs in adjC.items() for v in vs if u not in adjC[v])
    dP = bfs_graphe(ST[0], adjP)
    dC = bfs_graphe(OF[0], adjC)
    graphs[n] = (adjP, adjC)
    print(f"n={n} : permutoedre diametre = {max(dP.values())} | treillis : asymetries={asym}, "
          f"excentricite du centre tout-lie = {max(dC.values())} -> diametre = {2 * max(dC.values())}")

def recensement(ST, adjP, adjC):
    """Pour chaque paire ordonnee de chambres : le chemin par les murs est-il plus court ?"""
    raccourcis = egal = paires = 0
    exemple = None
    for a in ST:
        dperm = bfs_graphe(a, adjP)
        dtl = bfs_graphe(a, adjC)
        for b in ST:
            paires += 1
            if dtl[b] < dperm[b]:
                raccourcis += 1
                if exemple is None and b > a:
                    exemple = (a, b, dperm[b], dtl[b])
            elif dtl[b] == dperm[b]:
                egal += 1
    return paires, raccourcis, egal, exemple

t0 = time.time()
paires4, rc4, eg4, ex4 = recensement(ST4, *graphs[4])
print(f"Recensement 2x2 (modele ferme) : {paires4} paires | raccourcis = {rc4} | egalites = {eg4}")
paires6, rc6, eg6, ex6 = recensement(ST6, *graphs[6])
print(f"Recensement 3x2 (modele ferme) : {paires6} paires | raccourcis = {rc6} "
      f"({100 * rc6 / paires6:.1f} %) | egalites = {eg6}")
print(f"  exemple : {ex6[0]} -> {ex6[1]} : d_perm = {ex6[2]}, d_treillis = {ex6[3]}")
print(f"  ({time.time() - t0:.1f}s -- exhaustif, pas d'echantillonnage)")

Substrat 2x2 : ordres faibles = 75 | stricts = 24 | chambres = 576


Substrat 3x2 : ordres faibles = 4683 | stricts = 720 | chambres = 518400
n=4 : permutoedre diametre = 6 | treillis : asymetries=0, excentricite du centre tout-lie = 3 -> diametre = 6


n=6 : permutoedre diametre = 15 | treillis : asymetries=0, excentricite du centre tout-lie = 5 -> diametre = 10
Recensement 2x2 (modele ferme) : 576 paires | raccourcis = 0 | egalites = 48


Recensement 3x2 (modele ferme) : 518400 paires | raccourcis = 77760 (15.0 %) | egalites = 65520
  exemple : (1, 2, 3, 4, 5, 6) -> (1, 5, 6, 4, 3, 2) : d_perm = 9, d_treillis = 8
  (4.5s -- exhaustif, pas d'echantillonnage)


### Lecture du resultat — 15,0 % de raccourcis la ou il n'y en avait aucun

**Le verdict 2x2 ne survit pas au changement de substrat.**

- A **4 cases**, le recensement exhaustif des 576 paires de chambres donne **0 raccourci** :
  traverser les murs ne rend jamais le chemin plus court. Les diametres sont egaux
  (`6 = 6`), et l'arithmetique explique pourquoi : le gain net d'un traversee vaut
  `(n - 1)(n - 4) / 2`, soit **0** a `n = 4`.
- A **6 cases**, le meme modele, le meme code, un seul parametre change : **77 760 paires
  sur 518 400 raccourcissent**, soit **15,0 %**. Le diametre par cote se contracte de
  `15` a `10`, et le gain net theorique vaut `(6 - 1)(6 - 4) / 2 = 5`. Un exemple
  explicite est affiche : deux tables dont la distance dans le permutoedre est `9` mais
  seulement `8` par les murs.

**Ce qui survit, et ce qui ne survit pas.** Le **verdict** « la stratification ne comprime
pas la metrique » etait une propriete du petit substrat : il **ne survit pas**. Ce qui
survit, en revanche, c'est la **methode** qui produit le verdict — un constructeur qui
produit le chemin, un verificateur qui le re-derive et re-cherche la minimalite, un
recensement **exhaustif** plutot qu'un echantillon, et un changement de substrat qui ne
touche qu'un parametre. C'est exactement ce que le carnet
[Metajeux-06](Metajeux/Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb) met en avant :
ce n'est pas le chiffre qui est nouveau, c'est la **frontiere de la loi**, mesuree.

## Ce que le lake Lean prouve

Le carnet [Metajeux-03](Metajeux/Metajeux-03-Abstraction-a-Dette-Python.ipynb) fait
elaborer le module `Abstraction` du lake `game_theory_lean` et en affiche le verdict :

- `raffinement_aggrave_la_dette` : raffiner une partition **aggrave** la dette — l'ecart
  mesure dans le carnet n'est donc pas une coincidence de la chaine de partitions
  choisie ;
- la theoreme ne depend que de `propext` (aucun `sorry`, aucun axiome fort : le carnet
  source affiche la liste des axiomes et le compte des `sorry` du module).

Ce carnet **lit** ce resultat, il ne le re-preuve pas : le lake n'est pas construit ici, et
un resultat formel recopie sans etre rejoue n'est pas une preuve. Ce qui est rejoue ici,
c'est la **mesure** Python ; le module Lean en est le **contre-triage**.

La lecon de la sous-serie tient dans cette articulation : une mesure seule est un fait
local, une preuve seule est un fait formel, les deux qui **se recoupent** sont une
connaissance.

## Par ou y entrer

Trois parcours, selon ce que vous voulez faire.

**1. Le geste de representation en trois minutes.** [Metajeux-01](Metajeux/Metajeux-01-Open-Games-Lentilles-Python.ipynb) (les lois
d'une lentille et la composition des reponses) puis [Metajeux-02](Metajeux/Metajeux-02-Casser-la-Composition-Python.ipynb) (la casse par
le plateau multivoque), et revenez ici a l'experience 1 pour voir les trois situations
rejouees.

**2. Le prix de la compression.** [Metajeux-03](Metajeux/Metajeux-03-Abstraction-a-Dette-Python.ipynb) seul, puis l'experience 2 : la
courbe de dette et sa lecture des deux cotes.

**3. La preuve par la recherche.** [Metajeux-04](Metajeux/Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) (constructeur et verificateur
separes), [Metajeux-05](Metajeux/Metajeux-05-Temoins-Impossibilite-Python.ipynb) (le temoin d'impossibilite), [Metajeux-06](Metajeux/Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb) (le
second substrat), puis les experiences 3a et 3b : la certification, puis la refutation du
verdict par changement de substrat.

## Exercice 1 — La casse est proportionnelle au couplage

**Contexte.** L'ecart de deviation mesure a l'experience 1 vaut `+0.300` pour un couplage
`r = 0.3`. On a vu que le profil `(0, 0)` est refuse par le jeu compose, et que `(1, 1)`
est accepte.

**Objectif.** Ecrire une fonction `ecart_du_plus_petit_argmax(r)` qui renvoie l'ecart de
deviation du profil `(0, 0)` dans le jeu compose, pour un couplage `r` donne. Determiner
ensuite la valeur de `r` a partir de laquelle la composition naive echoue, et dire si cette
valeur depend de la taille du plateau ou du couplage.

**Indices.**

1. Reutilisez `u1`, `u2` et `br2` de l'experience 1 ; le seul parametre qui change est
   `R_COUPLAGE`.
2. L'ecart se lit comme `max_{x'} u1(x', br2(x')) - u1(0, br2(0))`.
3. Testez `r` dans `0.1, 0.2, ..., 0.9` et comparez la suite obtenue a `r` lui-meme.

In [6]:
# Exercice 1 a completer
# Etape 1 : ecrire ecart_du_plus_petit_argmax(r) qui renvoie l'ecart de deviation de (0, 0).
# Etape 2 : determiner le seuil de r au-dela duquel la composition naive echoue.
# Etape 3 : conclure sur ce que depend ce seuil.
# Indice : le seul parametre a faire varier est R_COUPLAGE ; comparez la suite obtenue a r.
print("Exercice a completer")
resultat_ex1 = None  # TODO etudiant : la formule de l'ecart, le seuil, et la conclusion

Exercice a completer


## Exercice 2 — Meme taille, autre dette

**Contexte.** A l'experience 2, la dette de l'abstraction a trois etats abstraits vaut
`1.9091` pour la partition `P3 = {{0,1}, {2,3}, {4,5}}`. La courbe de dette a un palier :
a quatre etats comme a trois etats, la dette est la meme.

**Objectif.** Construire une partition **hors chaine** de trois blocs, par exemple
`P3B = {{0,4}, {1,5}, {2,3}}`, resoudre l'abstrait exactement, retransporter, et mesurer
l'exploitabilite dans `G`. Comparer a celle de `P3`. Conclure : la dette depend-elle de la
seule taille de l'abstraction ?

**Indices.**

1. `solve_and_transport(partition)` et `exploitabilite(sig1, sig2)` de l'experience 2
   fonctionnent pour **toute** partition : la chaine n'etait qu'un choix.
2. `solve_and_transport` rend `(sig1, sig2, value_tilde)` ; `exploitabilite` rend
   `(expl, br1, br2)`.
3. La comparaison se fait a **taille egale** : les deux partitions ont trois blocs.

In [7]:
# Exercice 2 a completer
# Etape 1 : poser PARTITION_3B = [(0, 4), (1, 5), (2, 3)].
# Etape 2 : solve exact + retransport + exploitabilite dans G (memes fonctions que l'experience 2).
# Etape 3 : comparer a la dette de P3 (1.9091) et conclure.
# Indice : solve_and_transport et exploitabilite ne dependent pas de la partition.
print("Exercice a completer")
resultat_ex2 = None  # TODO etudiant : la dette de P3B et la comparaison a P3

Exercice a completer


## Exercice 3 — Les antipodes raccourcis

**Contexte.** A l'experience 3b, 15,0 % des paires de chambres 3x2 sont raccourcies par les
murs. Les paires les plus longues — les **antipodes**, a distance `15` dans le permutoedre
(le renversement d'une table) — concentrent sans doute la plupart de ces raccourcis.

**Objectif.** Pour chacune des 720 tables strictes de l'univers 3x2, mesurer la distance
de son renversement dans le permutoedre et dans le treillis, et compter combien de paires
antipodales sont raccourcies par les murs. Comparer ce compte au nombre total de
raccourcis (`77 760`) et conclure sur la repartition des raccourcis.

**Indices.**

1. Le renversement d'une table `t` est `tuple(reversed(t))`.
2. `bfs_graphe(a, adjP)` donne les distances depuis `a` dans le permutoedre ;
   `bfs_graphe(a, adjC)` dans le treillis — les deux sont deja construits dans `graphs[6]`.
3. Une paire est raccourcie si `d_treillis < d_perm` ; ici `d_perm` vaut toujours 15 pour
   une paire antipodale.

In [8]:
# Exercice 3 a completer
# Etape 1 : pour chaque table stricte de ST6, calculer son renversement.
# Etape 2 : mesurer d_perm et d_treillis vers ce renversement (bfs_graphe + graphs[6]).
# Etape 3 : compter les paires antipodales raccourcies et comparer au total (77760).
# Indice : une paire antipodale est raccourcie si d_treillis < 15.
print("Exercice a completer")
resultat_ex3 = None  # TODO etudiant : le compte des antipodes raccourcis et la repartition

Exercice a completer


## A retenir

- **Composer** ne survit qu'en **ensembles**. La correspondance des meilleures reponses se
  compose ; la selection d'un de ses elements ne se compose que si la reponse locale est
  univoque. Multivocite + couplage suffisent a briser la selection (`+0.300` d'ecart).
- **Abstraire** ne survit pas du tout a l'equilibre, mais la **dette** survit : elle est
  chiffree (`1.9091`), elle se lit des deux cotes (`+1.3961` / `+0.5130`), et elle est
  lisible des deux cotes *et* recoupeee par une preuve Lean.
- **Certifier** survit au changement de representation : le verdict d'impossibilite porte
  une structure (decomposition du produit), pas une mesure, et il se verifie par la
  recherche (`0` ecart sur 576 chambres, `13 824` triplets sans violation).
- **Une mesure** ne survit pas : `0` raccourci sur 576 paires a 4 cases, `15,0 %` sur
  518 400 paires a 6 cases. Le verdict « la stratification ne comprime pas la metrique »
  etait une propriete du petit substrat.

**Ce qui survit, en une phrase :** la **methode** (composer des correspondances, mesurer la
dette dans le jeu d'origine, certifier par la recherche, recenser exhaustivement) — et non
les nombres, qui sont des proprietes du substrat.

## Verifiez votre comprehension

1. Pourquoi la composition des **correspondances** survit-elle a la casse de la selection ?
2. Pourquoi la dette de l'abstraction doit-elle etre mesuree dans `G` et pas dans `G_tilde` ?
3. Une mesure qui change quand on change de substrat : qu'est-ce qui lui manque pour etre
   un resultat transportable ?

<details>
<summary>Elements de reponse</summary>

1. Parce qu'une correspondance ne choisit rien : elle rend l'ensemble des meilleures reponses, avec son contexte, et c'est la selection d'un element dans cet ensemble qui ne se compose pas. L'experience 1 le mesure : la reponse locale de J1 est le plateau `{0, 1}`, la composition des correspondances rend les deux profils `[(0, 0), (1, 1)]`, et la composition naive (plus petit argmax) propose `(0, 0)` — que le jeu compose refuse, avec un ecart de `+0.300` en faveur d'une deviation vers `x = 1`. Le profil `(1, 1)`, lui, est un equilibre du jeu compose. La lecture du resultat le formule ainsi : c'est une egalite d'ensembles, et la lecture fonctionnelle ne survit que si la reponse locale est univoque.
2. Parce que dans `G_tilde`, la strategie abstraite est un equilibre par construction : l'abstrait est resolu exactement, et sa dette y serait nulle. Ce qui est en question, c'est ce que vaut cette strategie une fois retransportee dans le jeu d'origine. La sortie le chiffre : `v(G) = -4.2143`, `v(G_tilde) = -1.8864`, et la strategie retransportee se fait exploiter des deux cotes — `BR_1 = -2.8182` (ecart `+1.3961`) et `BR_2 = 4.7273` (ecart `+0.5130`), soit une exploitabilite de `1.9091`. Mesurer dans `G_tilde` ne mesurerait que la qualite du solveur abstrait ; la dette est la perte subie dans `G`.
3. Il lui manque une structure qui la porte et qui se verifie independamment du substrat. Le carnet le montre par contraste : le verdict `IMPOSSIBLE` survit parce qu'il s'appuie sur la decomposition du produit — `d_chambre = d_perm(row) + d_perm(col)`, verifiee sans ecart sur `576` chambres — tandis que la mesure ne survit pas : `0` raccourci sur `576` paires a 4 cases, `77 760` sur `518 400` a 6 cases (`15,0 %`), avec des diametres `6 = 6` a 4 cases contre `15` a `10` a 6 cases. Ce qui se transporte, c'est la methode qui produit le verdict — constructeur, verificateur, recensement exhaustif, changement d'un seul parametre — et non le nombre, qui reste une propriete du substrat. C'est la conclusion de l'auto-verdict : la methode survit au changement de representation, pas les mesures.

</details>

## Pour aller plus loin

- Le carnet [Metajeux-04](Metajeux/Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb)
  detaille le constructeur et le verificateur separes, et le controle negatif qui les
  distingue.
- Le carnet [Metajeux-06](Metajeux/Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb)
  detaille le lemme de decomposition du produit de jeux, et la loi du troc
  `(n - 1)(n - 4) / 2` qui borne la frontiere 2x2 / 3x2.

In [9]:
# Controle de coherence interne : le notebook se ferme sur son auto-verdict.
# Les valeurs sont CALCULEES depuis les cellules precedentes, jamais recopiees ;
# chaque assertion les confronte aux valeurs des carnets sources.
ecart_naif = max(u1(xp, br2(xp)) for xp in ACTIONS_X) - u1(selection_naive, br2(selection_naive))
contre_fait_univoque_ok = est_nash_compose(x_u, y_u, u1_univoque)
contre_fait_r0_ok = all(abs(e) < 1e-9 for e in ecarts_r0)
dette_p3 = expl  # exploitabilite de la strategie retransportee (P3)
verdicts_3a = [certifier(g, h, k)["verdict"] for _, g, h, k in
                (("CAS A", IDENTITE, RENVERS, 5),
                 ("CAS B", IDENTITE, H_B, 3),
                 ("CAS C", IDENTITE, H_C, 1))]
n_impossible = verdicts_3a.count("IMPOSSIBLE")
n_possible = verdicts_3a.count("POSSIBLE")

# --- Assertions contre les valeurs des carnets sources ---
assert abs(ecart_naif - 0.300) < 1e-9, "experience 1 : l'ecart du carnet source a change"
assert contre_fait_univoque_ok and contre_fait_r0_ok, "experience 1 : un contre-fait ne tient plus"
assert abs(VALUE_G + 4.2143) < 5e-5, "experience 2 : v(G) ne vaut plus la valeur du carnet source"
assert abs(dette_p3 - 1.9091) < 5e-5, "experience 2 : la dette d'abstraction a change"
assert plateau, "experience 2 : le palier P4 = P3 ne tient plus"
assert len(chambres) == 576 and not ecarts, "experience 3a : la structure produit n'est plus exacte"
assert (n_impossible, n_possible) == (2, 1), "experience 3a : les verdicts ont change"
assert seuil == 12, "experience 3a : la bascule du verdict a bouge"
assert (len(atte_A), len(atte_B), len(atte_C)) == (235, 68, 7), "experience 3a : le BFS tronque a change"
assert (paires4, rc4) == (576, 0), "experience 3b : le recensement 2x2 a change"
assert (paires6, rc6) == (518400, 77760), "experience 3b : le recensement 3x2 a change"

print("GT-18 : Capstone Metajeux -- ce qui survit au changement de representation")
print()
print("Coherence interne (valeurs calculees, comparees aux carnets sources) :")
print(f"  - Experience 1 : composition naive NON-NASH (ecart {ecart_naif:+.3f}) ; "
      f"contre-faits univoque et r=0 : OK")
print(f"  - Experience 2 : v(G) = {VALUE_G:.4f} ; dette d'abstraction = {dette_p3:.4f} "
      f"(palier P4 = P3)")
print(f"  - Experience 3a : structure produit EXACTE sur {len(chambres)} chambres ; "
      f"{n_impossible} IMPOSSIBLE, {n_possible} POSSIBLE ; bascule a k={seuil}")
print(f"  - Experience 3b : {rc4} raccourci sur {paires4} paires (2x2) ; "
      f"{rc6} sur {paires6} (3x2, {100 * rc6 / paires6:.1f} %)")
print("  - Conclusion : la methode survit au changement de representation, pas les mesures.")


GT-18 : Capstone Metajeux -- ce qui survit au changement de representation

Coherence interne (valeurs calculees, comparees aux carnets sources) :
  - Experience 1 : composition naive NON-NASH (ecart +0.300) ; contre-faits univoque et r=0 : OK
  - Experience 2 : v(G) = -4.2143 ; dette d'abstraction = 1.9091 (palier P4 = P3)
  - Experience 3a : structure produit EXACTE sur 576 chambres ; 2 IMPOSSIBLE, 1 POSSIBLE ; bascule a k=12
  - Experience 3b : 0 raccourci sur 576 paires (2x2) ; 77760 sur 518400 (3x2, 15.0 %)
  - Conclusion : la methode survit au changement de representation, pas les mesures.


## References

**Ghani, N., Hedges, J., Winschel, V. & Zahn, P. (2018)** — *Compositional Game Theory*,
LICS '18, arXiv:1603.04641. -> l'ancrage de [Metajeux-01](Metajeux/Metajeux-01-Open-Games-Lentilles-Python.ipynb) :
la composition des open games preserve la structure de meilleure reponse. Le carnet
source en verifie numeriquement le cas d'usage ; l'experience 1 de ce carnet rejoue le
contre-exemple (multivocite + couplage) ou la lecture fonctionnelle echoue.

**Hedges, J. (2017)** — *Coherence for lenses and open games*, arXiv:1704.02230. ->
[lentes et lois `get-put` / `put-get`](Metajeux/Metajeux-01-Open-Games-Lentilles-Python.ipynb),
la semantique des lentilles coherentes dont depend le regime univoque de l'experience 1.

**Kroer, M. & Sandholm, T.** — *Extensive-form game solving with abstraction*, et le
travail de Sandholm sur l'approximation par partition (postes par le carnet
[Metajeux-03](Metajeux/Metajeux-03-Abstraction-a-Dette-Python.ipynb)). -> la **borne** de
l'exploitabilite d'une abstraction. Le carnet source signale explicitement qu'elle est
**rapportee** et non redemontree : ce carnet fait de meme, et s'en tient a la **mesure**.

**Robinson, R. & Goforth, W.** — les jeux ordinaux 2x2 (premiere attestation du modele,
[Metajeux-04](Metajeux/Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb)), dont le
carnet [Metajeux-06](Metajeux/Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb) etend
le modele au substrat 3x2 ordinal.